In [1]:
from StrategyExecutor import fundamental_strat
import os
import os.path as op
from datetime import datetime, timedelta
import time
import pandas as pd

from DataHandler import dgp_config as dcf  # must define STOCKS_SAVEPATH

In [2]:
fund = fundamental_strat.get_fundamental_weights()
fund = fund.dropna()

In [3]:
save_date = datetime.today().strftime("%Y%m%d")
tech_dir = op.join(dcf.WORK_DIR, "current", save_date, 'ensem5_res_current.csv')
tech = pd.read_csv(tech_dir)
# Rank 'tech' DataFrame by 'prob_up' in descending order and create a new column 'tech_rank'
tech['tech_rank'] = tech['up_prob'].rank(ascending=False, method='min')

# Display the top rows of the DataFrame with the new 'tech_rank' column
print(tech.head())


   Unnamed: 0  StockID ending_date   up_prob  ret_val  MarketCap  tech_rank
0           0        1  2099-12-31  0.597631      0.0        0.0      141.0
1           1        2  2099-12-31  0.509785      0.0        0.0      979.0
2           2        6  2099-12-31  0.366763      0.0        0.0     4806.0
3           3        7  2099-12-31  0.554831      0.0        0.0      373.0
4           4        8  2099-12-31  0.442091      0.0        0.0     2989.0


In [4]:
# Convert both 'StockID' columns to string
tech['StockID'] = tech['StockID'].astype(str).str.zfill(6)
fund['StockID'] = fund['StockID'].astype(str).str.zfill(6)

print(tech.columns)
print(fund.columns)

Index(['Unnamed: 0', 'StockID', 'ending_date', 'up_prob', 'ret_val',
       'MarketCap', 'tech_rank'],
      dtype='object')
Index(['StockID', 'fund_rank', 'pi', 'pe', 'pb', 'debt-to-asset'], dtype='object')


In [5]:
# Perform the join based on the 'StockID' column
merged_df = pd.merge(tech, fund, how='inner', on='StockID')

# Display the merged dataframe
print(merged_df.head())

   Unnamed: 0 StockID ending_date   up_prob  ret_val  MarketCap  tech_rank  \
0           3  000007  2099-12-31  0.554831      0.0        0.0      373.0   
1           5  000009  2099-12-31  0.444009      0.0        0.0     2921.0   
2          11  000017  2099-12-31  0.413161      0.0        0.0     3907.0   
3          12  000019  2099-12-31  0.375999      0.0        0.0     4675.0   
4          13  000020  2099-12-31  0.366909      0.0        0.0     4802.0   

   fund_rank        pi            pe         pb  debt-to-asset  
0       22.0  0.494072    268.892785  13.932404      39.915671  
1     1520.0  4.897501  12536.315955   2.420353      59.617126  
2       20.0   0.47185    248.693436  12.680398      15.263579  
3     1029.0  3.705257     24.384226   1.598527      36.206559  
4       73.0   1.06727    109.045226   9.243227      40.655197  


In [6]:
merged_df['tech_rank'] = pd.to_numeric(merged_df['tech_rank'])
merged_df['fund_rank'] = pd.to_numeric(merged_df['fund_rank'])
merged_df['merge_rank'] = merged_df['tech_rank'] + merged_df['fund_rank']

In [7]:
merged_df_sorted = merged_df.sort_values(by='merge_rank', ascending=True)
# Create a boolean column 'top_10_percent' based on 'tech_rank' and 'fund_rank' being less than or equal to 500
merged_df_sorted['top_10_percent'] = (merged_df_sorted['tech_rank'] <= 500) & (merged_df_sorted['fund_rank'] <= 500)
# Get the top 100 rows from merged_df_sorted
top_100_df = merged_df_sorted.head(100)
# Count how many rows have 'top_10_percent' as True in the top 100 rows
top_100_true_count = top_100_df['top_10_percent'].sum()

# Display the count
print(f"Number of top 100 rows where 'top_10_percent' is True: {top_100_true_count}")



Number of top 100 rows where 'top_10_percent' is True: 61


In [8]:
merged_df_sorted.head()
output_file_path = 'sorted_merged_df.csv'

# Save the sorted DataFrame to a CSV file
merged_df_sorted.to_csv(output_file_path, index=False)